<a href="https://colab.research.google.com/github/rijal0708/data-science-2026/blob/main/Pertemuan10_M_Rijal_Anshory_DJ_240401010281_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama    = M Rijal Anshory DJ.

NIM     = 240401010281

Kelas   = IF403

---



1. muat dan eksplorasi data

In [2]:
from google.colab import files

uploaded = files.upload()
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Ukuran Dataset:",df.shape)
print(df.info())
print(df.head())
print("\nProporsi Kelas Churn")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
Ukuran Dataset: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 no

2. preprocessing

In [4]:
from sklearn.model_selection import train_test_split

df = df.drop("customerID", axis=1)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df = pd.get_dummies(df, drop_first=True) # one hot encoding

# Memisahkan fitur dan target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Train-test split dengan stratify
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Data Latih :", X_tr.shape)
print("Data Uji   :", X_te.shape)


Data Latih : (5634, 30)
Data Uji   : (1409, 30)


3. latih model

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

4. evaluasi model

In [6]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

# prediksi kelas
y_pred = rf.predict(X_te)

# prediksi probabilitas kelas
y_prob = rf.predict_proba(X_te)[:,1]

print("Accuracy :", accuracy_score(y_te, y_pred))

print("\nConfusion Matrix ")
print(confusion_matrix(y_te, y_pred))

print("\nClassification Report")
print(classification_report(y_te, y_pred))

print("\nROC AUC Score")
print(roc_auc_score(y_te, y_prob))


Accuracy : 0.7892122072391767

Confusion Matrix 
[[925 110]
 [187 187]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


ROC AUC Score
0.8246208891988943


5. prediksi probabilitas churn

In [9]:
hasil = X_te.copy()

hasil["Actual"] = y_te.values
hasil["Prediksi"] = y_pred
hasil["Probabilitas Churn"] = y_prob

hasil = hasil.sort_values("Probabilitas Churn", ascending=False)
hasil.head(10)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Actual,Prediksi,Probabilitas Churn
1731,1,1,69.60,69.60,False,False,False,True,False,False,...,False,False,False,True,False,True,False,1,1,1.000000
2194,0,1,79.50,79.50,True,False,False,True,False,False,...,False,False,False,True,False,True,False,1,1,0.993333
809,0,1,69.55,69.55,True,False,False,True,False,False,...,False,False,False,True,False,True,False,1,1,0.990000
6623,1,1,76.45,76.45,True,False,False,True,False,True,...,False,False,False,True,False,True,False,1,1,0.990000
1739,0,1,69.90,69.90,True,False,False,True,False,False,...,False,False,False,True,False,True,False,1,1,0.986667
2927,0,1,69.90,69.90,True,False,False,True,False,False,...,False,False,False,True,False,True,False,0,1,0.986667
3346,1,2,84.05,186.05,False,False,False,True,False,True,...,True,False,False,True,False,True,False,0,1,0.970000
1144,0,1,35.55,35.55,True,False,False,False,True,False,...,True,False,False,True,False,True,False,1,1,0.963333
4585,1,1,85.05,85.05,False,False,False,True,False,True,...,True,False,False,True,False,True,False,1,1,0.960000
2729,0,2,85.70,169.80,False,False,False,True,False,True,...,True,False,False,True,False,True,False,1,1,0.950000


6. kesimpulan

Berdasarkan hasil pelatihan menggunakan Random Forest dengan class_weight="balanced", model mampu mengklasifikasikan pelanggan yang berpotensi melakukan churn dengan performa yang baik. Penggunaan class_weight="balanced" membantu model menangani ketidakseimbangan jumlah data antara pelanggan churn dan non-churn sehingga kemampuan mendeteksi pelanggan churn meningkat. Nilai ROC-AUC yang tinggi menunjukkan bahwa model memiliki kemampuan yang baik dalam membedakan pelanggan yang akan churn dan yang tidak. Selain itu, probabilitas yang dihasilkan oleh predict_proba() dapat dimanfaatkan oleh perusahaan untuk mengidentifikasi pelanggan dengan risiko churn tinggi sehingga dapat dilakukan tindakan pencegahan, seperti pemberian promosi atau peningkatan kualitas layanan.